In [1]:
# 6-21-2026

In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
import glob
import os

In [2]:
def prep_features(X_raw, scaler):
    # scales with the domain's shared scaler, then drops the redundant lccs column for nn input only
    X_scaled = pd.DataFrame(scaler.transform(X_raw), columns=X_raw.columns, index=X_raw.index)
    return X_scaled.drop(columns=["lccs_class_2"]).values

In [3]:
def build_model(input_dim):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(32, activation="relu"),
        layers.Dense(32, activation="relu"),
        layers.Dense(1)
    ])
# simple model since rf/xgb showed low signal cieling
    model.compile(optimizer="adam", loss="mse")
    return model

In [34]:
domain_id = "22"

X_train = pd.read_csv(f"train_X/domain_{domain_id}.csv")
y_train = pd.read_csv(f"train_y/domain_{domain_id}.csv")["log_ba"]
X_test = pd.read_csv(f"test_X/domain_{domain_id}.csv")
y_test = pd.read_csv(f"test_y/domain_{domain_id}.csv")["log_ba"]

scaler = joblib.load(f"scalers/domain_{domain_id}.joblib")
X_train_nn = prep_features(X_train, scaler)
X_test_nn = prep_features(X_test, scaler)

In [35]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_nn, y_train, test_size=0.2, random_state=5
)

In [4]:
NN_PARAMS = {
    "epochs": 100,
    "batch_size": 256,
    "patience": 10 # if loss doesnt imporve over this many epochs, stop
}

In [5]:
tf.random.set_seed(10)
model = build_model(input_dim=X_tr.shape[1])

early_stop = callbacks.EarlyStopping(
    monitor="val_loss",
    patience=NN_PARAMS["patience"],
    restore_best_weights=True
)

history = model.fit(
    X_tr, y_tr,
    validation_data=(X_val, y_val),
    epochs=NN_PARAMS["epochs"],
    batch_size=NN_PARAMS["batch_size"],
    callbacks=[early_stop],
    verbose=0
)

NameError: name 'X_tr' is not defined

In [38]:
print(f"stopped at epoch: {len(history.history['loss'])}")

stopped at epoch: 69


In [39]:
val_pred = model.predict(X_val, verbose=0).flatten()
test_pred = model.predict(X_test_nn, verbose=0).flatten()
train_pred = model.predict(X_tr, verbose=0).flatten()


val_spearman, _ = spearmanr(y_val, val_pred)
test_spearman, _ = spearmanr(y_test, test_pred)
train_spearman, _ = spearmanr(y_tr, train_pred)


print(f"val spearman: {val_spearman:.4f}, test spearman: {test_spearman:.4f}, train spearman: {train_spearman:.4f}")

val spearman: 0.3505, test spearman: 0.3462, train spearman: 0.3840


In [40]:
# initial expereimenting done, now making T_nn

In [6]:
domain_files = glob.glob("train_X/domain_*.csv")
domain_ids = sorted(
    int(os.path.basename(f).replace("domain_", "").replace(".csv", ""))
    for f in domain_files
)
print(f"found {len(domain_ids)} domains")

found 34 domains


In [7]:
models = {}
scalers = {}

In [8]:
for domain_id in domain_ids:
    X_train = pd.read_csv(f"train_X/domain_{domain_id}.csv")
    y_train = pd.read_csv(f"train_y/domain_{domain_id}.csv")["log_ba"]
    # setup train/test/scalers
    scaler = joblib.load(f"scalers/domain_{domain_id}.joblib")
    X_train_nn = prep_features(X_train, scaler)

    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train_nn, y_train, test_size=0.2, random_state=5
    ) # val split

    keras.backend.clear_session()
    tf.random.set_seed(10)
    model = build_model(input_dim=X_tr.shape[1])

    early_stop = callbacks.EarlyStopping(
        monitor="val_loss",
        patience=NN_PARAMS["patience"],
        restore_best_weights=True
    )

    model.fit(
        X_tr, y_tr,
        validation_data=(X_val, y_val),
        epochs=NN_PARAMS["epochs"],
        batch_size=NN_PARAMS["batch_size"],
        callbacks=[early_stop],
        verbose=0
    )

    models[domain_id] = model
    scalers[domain_id] = scaler

    print(f"domain {domain_id} done, stopped at epoch {len(model.history.history['loss'])}")
# takes ~30 min


domain 0 done, stopped at epoch 70
domain 1 done, stopped at epoch 52
domain 2 done, stopped at epoch 54
domain 4 done, stopped at epoch 73
domain 5 done, stopped at epoch 48
domain 6 done, stopped at epoch 47
domain 7 done, stopped at epoch 88
domain 8 done, stopped at epoch 55
domain 11 done, stopped at epoch 71
domain 12 done, stopped at epoch 68
domain 13 done, stopped at epoch 68
domain 16 done, stopped at epoch 52
domain 18 done, stopped at epoch 56
domain 19 done, stopped at epoch 75
domain 20 done, stopped at epoch 100
domain 21 done, stopped at epoch 34
domain 22 done, stopped at epoch 87
domain 23 done, stopped at epoch 57
domain 25 done, stopped at epoch 37
domain 26 done, stopped at epoch 59
domain 27 done, stopped at epoch 65
domain 28 done, stopped at epoch 37
domain 29 done, stopped at epoch 67
domain 30 done, stopped at epoch 92
domain 32 done, stopped at epoch 51
domain 33 done, stopped at epoch 69
domain 36 done, stopped at epoch 53
domain 37 done, stopped at epoch 8

In [9]:
test_X_raw = {}
test_y = {}

In [10]:
for domain_id in domain_ids:
    test_X_raw[domain_id] = pd.read_csv(f"test_X/domain_{domain_id}.csv")
    test_y[domain_id] = pd.read_csv(f"test_y/domain_{domain_id}.csv")["log_ba"]

In [11]:
T_spearman_nn = pd.DataFrame(index=domain_ids, columns=domain_ids, dtype=float)

In [12]:
for i in domain_ids:
    model_i = models[i]
    scaler_i = scalers[i]

    for j in domain_ids:
        # apply source domain's scaler to target domain's raw test X, not target's own scaler
        X_test_nn = prep_features(test_X_raw[j], scaler_i)
        y_true = test_y[j]

        preds = model_i.predict(X_test_nn, verbose=0).flatten()
        T_spearman_nn.loc[i, j], _ = spearmanr(y_true, preds)

    print(f"evaluated source domain {i} against all targets")
# takes ~20 min

evaluated source domain 0 against all targets
evaluated source domain 1 against all targets
evaluated source domain 2 against all targets
evaluated source domain 4 against all targets
evaluated source domain 5 against all targets
evaluated source domain 6 against all targets
evaluated source domain 7 against all targets
evaluated source domain 8 against all targets
evaluated source domain 11 against all targets
evaluated source domain 12 against all targets
evaluated source domain 13 against all targets
evaluated source domain 16 against all targets
evaluated source domain 18 against all targets
evaluated source domain 19 against all targets
evaluated source domain 20 against all targets
evaluated source domain 21 against all targets
evaluated source domain 22 against all targets
evaluated source domain 23 against all targets
evaluated source domain 25 against all targets
evaluated source domain 26 against all targets
evaluated source domain 27 against all targets
evaluated source doma

In [13]:
T_spearman_nn

,0,1,2,4,5,6,7,8,11,12,...,32,33,36,37,38,39,45,46,47,49
0,0.320092,0.218506,0.010315,0.136371,0.206335,0.083833,0.123209,-0.040062,0.203783,0.037110,...,0.172640,0.217539,0.143141,0.282709,0.216132,-0.021657,0.016860,0.042688,-0.045014,0.032029
1,0.185511,0.301423,0.053854,0.063554,0.173079,0.030248,0.063665,-0.012455,0.186188,0.042191,...,0.138469,0.080097,0.119579,0.241250,0.091713,0.003888,0.037578,-0.057061,0.130706,0.034165
2,-0.039632,0.124095,0.272463,-0.011210,-0.073167,0.015664,-0.116147,0.035717,-0.059385,0.012884,...,0.046300,0.050518,0.039031,-0.019641,-0.126936,0.049768,-0.106521,0.122971,0.054498,-0.020650
4,0.207405,0.176480,0.162360,0.312562,0.113447,0.177426,0.151855,0.031251,0.132639,0.034974,...,0.166936,0.213393,0.164746,0.209929,0.209517,-0.013279,0.043378,0.259744,0.102719,-0.000712
5,0.202683,0.142978,0.077551,0.157044,0.371449,0.065264,0.078839,0.121912,0.215937,0.033538,...,0.110173,-0.069247,0.100397,0.251185,0.140583,-0.014240,0.063131,0.028018,0.050406,0.099768
6,-0.048291,0.012234,0.028097,0.182366,0.008454,0.238876,0.042646,-0.143914,0.060181,0.037157,...,0.047783,0.139531,0.087209,-0.038671,0.121483,0.045190,0.105016,0.211258,0.105505,0.019181
7,0.044555,-0.013462,0.002172,0.164467,0.060079,0.116029,0.249115,-0.042529,0.085797,0.121324,...,0.040926,0.146981,0.083670,0.011314,0.171518,-0.002957,-0.004960,0.102449,0.030155,-0.019714
8,0.036366,0.154760,0.049130,0.137715,0.065093,0.093301,0.045075,0.336984,0.025888,-0.094570,...,0.008310,-0.010210,0.003283,0.048352,0.003244,0.026047,-0.078929,0.110994,0.043491,0.029830
11,0.198182,0.163577,0.015697,0.076206,0.208980,0.076158,0.100152,0.023031,0.446144,0.031662,...,0.249679,0.104526,0.112799,0.344951,0.139751,0.016878,0.117567,0.041276,0.013937,0.009602
12,0.059336,0.163136,-0.073969,0.156336,0.128524,0.113191,0.024515,0.030718,0.118413,0.382851,...,0.052816,0.054698,0.029302,0.147441,0.049187,-0.026733,0.016606,0.177426,0.055362,0.046877


In [14]:
T_spearman_nn.to_csv("transfer_matrix_spearman_nn_10.csv")